# 02 — Flow Head Pretraining

Trains a **MeshGraphNet** surrogate on 400 AirfRANS cases to predict:
- `u_x`, `u_y` — velocity components
- `p_over_rho` — kinematic pressure
- `nu_t` — turbulent viscosity

**Requires:** A100 GPU + `00_env_setup.ipynb` already run this session.

**Checkpoint saved to:** `Drive/cfd-ald-app/checkpoints/flow_head/`

## 0. Mount Drive + clone/pull repo

In [ ]:
# Install all dependencies — required every new Colab session
import subprocess, torch

cuda_ver = torch.version.cuda or ''
extra = 'cu13' if cuda_ver.startswith('13') else 'cu12'
print(f'Installing nvidia-physicsnemo[{extra},nn-extras] ...')
subprocess.run(['pip', 'install', '-q', f'nvidia-physicsnemo[{extra},nn-extras]'], check=True)
subprocess.run(['pip', 'install', '-q', 'torch_geometric', 'h5py', 'scipy', 'tqdm'], check=True)
print('Done.')

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = '/content/drive/MyDrive/cfd-ald-app'
except ImportError:
    BASE_DIR = os.path.expanduser('~/cfd-ald-app-data')

REPO_URL = 'https://github.com/Ranaam21/cfd-ald-app.git'
REPO_DIR = '/content/cfd-ald-app'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or 'Repo up to date.')
else:
    r = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'git clone failed: {r.stderr}')
    print('Cloned.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

AIRFRANS_PROC = Path(BASE_DIR) / 'data' / 'processed' / 'airfrans'
CKPT_DIR      = Path(BASE_DIR) / 'checkpoints' / 'flow_head'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Processed data → {AIRFRANS_PROC}')
print(f'Checkpoints    → {CKPT_DIR}')

## 1. Imports + GPU check

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import h5py
import json
import time
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Tuple, Optional
from tqdm import tqdm
from scipy.spatial import cKDTree

assert torch.cuda.is_available(), 'GPU required — switch runtime to A100'
DEVICE = torch.device('cuda')
print(f'PyTorch {torch.__version__}  |  {torch.cuda.get_device_name(0)}')

import torch_geometric
from torch_geometric.nn import MessagePassing
print(f'PyG {torch_geometric.__version__}: OK')
print('Imports ready.')

## 2. Config

In [ ]:
CFG = {
    # Data
    'train_split':    'scarce/train',
    'test_split':     'scarce/test',
    'max_nodes':      80_000,    # subsample large graphs to this size

    # Graph
    'k_neighbors':    6,         # k-NN edges per node

    # Model
    'input_dim_nodes': 17,       # coords(3) + node_feats(3) + global_broadcast(11)
    'input_dim_edges': 4,        # dx, dy, dz=0, dist
    'output_dim':      4,        # u_x, u_y, p_over_rho, nu_t
    'processor_size':  15,       # message passing steps
    'hidden_dim':      256,

    # Training
    'epochs':          100,
    'lr':              3e-4,
    'lr_min':          1e-5,
    'grad_accum':      4,        # effective batch size = 4 graphs
    'grad_clip':       1.0,
    'bf16':            True,     # bfloat16 on A100

    # Saving
    'save_every':      10,       # save checkpoint every N epochs
    'log_every':       5,
}

print('Config:')
for k, v in CFG.items():
    print(f'  {k:20s} = {v}')

## 3. Dataset

In [ ]:
class AirfRANSCase:
    """Lazy loader for one AirfRANS HDF5 case."""

    def __init__(self, h5_path: Path, max_nodes: int = None):
        self.path = h5_path
        self.max_nodes = max_nodes

    def load(self) -> dict:
        with h5py.File(self.path, 'r') as h5:
            coords       = h5['coords'][:]               # [N, 3]
            node_feats   = h5['inputs/node_features'][:] # [N, 3]
            global_feats = h5['inputs/global'][:]        # [11]
            targets      = h5['outputs/node_fields'][:]  # [N, 4]

        N = len(coords)
        if self.max_nodes and N > self.max_nodes:
            idx = np.random.choice(N, self.max_nodes, replace=False)
            idx.sort()
            coords, node_feats, targets = coords[idx], node_feats[idx], targets[idx]

        # Broadcast global features to every node
        G = global_feats.shape[0]
        global_broadcast = np.tile(global_feats, (len(coords), 1))   # [N, 11]

        node_input = np.concatenate([coords, node_feats, global_broadcast], axis=1)  # [N, 17]
        return {'node_input': node_input, 'coords': coords, 'targets': targets}


def load_split(proc_dir: Path, split: str, max_nodes: int) -> List[AirfRANSCase]:
    split_dir = proc_dir / split
    h5_files  = sorted(split_dir.rglob('*.h5'))
    print(f'  {split}: {len(h5_files)} cases')
    return [AirfRANSCase(p, max_nodes) for p in h5_files]


print('Loading file lists...')
train_cases = load_split(AIRFRANS_PROC, CFG['train_split'], CFG['max_nodes'])
test_cases  = load_split(AIRFRANS_PROC, CFG['test_split'],  CFG['max_nodes'])
print(f'Total: {len(train_cases)} train, {len(test_cases)} test')

## 4. Normalisation stats

Compute mean/std over a sample of training cases so inputs and outputs are ~N(0,1).

In [ ]:
norm_path = CKPT_DIR / 'norm_stats.npz'

if norm_path.exists():
    stats = np.load(norm_path)
    node_mean = stats['node_mean']; node_std = stats['node_std']
    out_mean  = stats['out_mean'];  out_std  = stats['out_std']
    print(f'Loaded norm stats from {norm_path}')
else:
    print('Computing normalisation stats over 50 training cases...')
    sample = train_cases[:50]
    all_nodes, all_outs = [], []
    for c in tqdm(sample):
        d = c.load()
        all_nodes.append(d['node_input'])
        all_outs.append(d['targets'])

    all_nodes = np.concatenate(all_nodes, axis=0)
    all_outs  = np.concatenate(all_outs,  axis=0)

    node_mean = all_nodes.mean(axis=0).astype(np.float32)
    node_std  = all_nodes.std(axis=0).clip(1e-6).astype(np.float32)
    out_mean  = all_outs.mean(axis=0).astype(np.float32)
    out_std   = all_outs.std(axis=0).clip(1e-6).astype(np.float32)

    np.savez(norm_path,
             node_mean=node_mean, node_std=node_std,
             out_mean=out_mean,   out_std=out_std)
    print(f'Saved to {norm_path}')

print('Output std (u_x, u_y, p_over_rho, nu_t):', out_std.round(4))

## 5. Graph builder

In [ ]:
def build_edges(coords: np.ndarray, k: int = 6) -> Tuple[np.ndarray, np.ndarray]:
    """k-NN graph via scipy — returns (src, dst) edge index arrays."""
    tree = cKDTree(coords[:, :2])
    _, idx = tree.query(coords[:, :2], k=k+1)
    idx = idx[:, 1:]   # drop self
    src = np.repeat(np.arange(len(coords)), k)
    dst = idx.flatten()
    return src, dst


def make_batch(coords: np.ndarray, node_input: np.ndarray, targets: np.ndarray,
               k: int, node_mean, node_std, out_mean, out_std) -> dict:
    """Build normalised PyG-style batch dict for one case."""
    src, dst = build_edges(coords, k)

    diff     = coords[dst] - coords[src]                          # [E, 3]
    dist     = np.linalg.norm(diff, axis=1, keepdims=True)       # [E, 1]
    med_dist = float(np.median(dist)) + 1e-8
    edge_feats = np.concatenate([diff / med_dist, dist / med_dist], axis=1).astype(np.float32)  # [E, 4]

    node_norm = ((node_input - node_mean) / node_std).astype(np.float32)
    tgt_norm  = ((targets    - out_mean)  / out_std).astype(np.float32)

    edge_index = torch.tensor(np.stack([src, dst], axis=0), dtype=torch.long)
    return {
        'x':          torch.from_numpy(node_norm),
        'edge_index': edge_index,
        'edge_attr':  torch.from_numpy(edge_feats),
        'y':          torch.from_numpy(tgt_norm),
    }


# Quick test
print('Testing graph builder...')
d = train_cases[0].load()
b = make_batch(d['coords'], d['node_input'], d['targets'],
               CFG['k_neighbors'], node_mean, node_std, out_mean, out_std)
print(f'  x:          {b["x"].shape}')
print(f'  edge_index: {b["edge_index"].shape}')
print(f'  edge_attr:  {b["edge_attr"].shape}')
print(f'  y:          {b["y"].shape}')
del d, b

## 6. Model

In [ ]:
def mlp(in_dim: int, out_dim: int, hidden: int = 256, layers: int = 2) -> nn.Sequential:
    dims = [in_dim] + [hidden] * (layers - 1) + [out_dim]
    mods = []
    for i in range(len(dims) - 1):
        mods.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims) - 2:
            mods.append(nn.SiLU())
    return nn.Sequential(*mods)


class MGNProcessor(MessagePassing):
    """One MeshGraphNet message-passing step."""

    def __init__(self, hidden: int):
        super().__init__(aggr='sum')
        self.edge_mlp = mlp(3 * hidden, hidden)
        self.node_mlp = mlp(2 * hidden, hidden)
        self.edge_norm = nn.LayerNorm(hidden)
        self.node_norm = nn.LayerNorm(hidden)

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        # Edge update
        new_e = self.edge_mlp(torch.cat([x[src], x[dst], edge_attr], dim=-1))
        new_e = self.edge_norm(new_e + edge_attr)   # residual
        # Node update (aggregate incoming edge messages)
        agg   = self.propagate(edge_index, x=x, edge_attr=new_e)
        new_x = self.node_mlp(torch.cat([x, agg], dim=-1))
        new_x = self.node_norm(new_x + x)           # residual
        return new_x, new_e

    def message(self, edge_attr):
        return edge_attr


class MeshGraphNet(nn.Module):
    """MeshGraphNet — encoder / N×processor / decoder."""

    def __init__(self, node_dim: int, edge_dim: int, out_dim: int,
                 hidden: int = 256, n_layers: int = 15):
        super().__init__()
        self.node_enc = mlp(node_dim, hidden)
        self.edge_enc = mlp(edge_dim, hidden)
        self.processors = nn.ModuleList([MGNProcessor(hidden) for _ in range(n_layers)])
        self.decoder = mlp(hidden, out_dim, hidden)

    def forward(self, x, edge_index, edge_attr):
        x = self.node_enc(x)
        e = self.edge_enc(edge_attr)
        for proc in self.processors:
            x, e = proc(x, edge_index, e)
        return self.decoder(x)


model = MeshGraphNet(
    node_dim  = CFG['input_dim_nodes'],
    edge_dim  = CFG['input_dim_edges'],
    out_dim   = CFG['output_dim'],
    hidden    = CFG['hidden_dim'],
    n_layers  = CFG['processor_size'],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'MeshGraphNet (PyG): {n_params/1e6:.1f}M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=CFG['lr_min']
)
scaler = torch.amp.GradScaler('cuda')
dtype  = torch.bfloat16 if CFG['bf16'] else torch.float32
print(f'Optimizer: AdamW  |  Scheduler: CosineAnnealing  |  dtype: {dtype}')

## 7. Resume from checkpoint (optional)

In [ ]:
start_epoch = 0
train_losses, val_losses = [], []

latest_ckpt = sorted(CKPT_DIR.glob('ckpt_epoch*.pt'))
if latest_ckpt:
    ckpt = torch.load(latest_ckpt[-1], map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch  = ckpt['epoch'] + 1
    train_losses = ckpt.get('train_losses', [])
    val_losses   = ckpt.get('val_losses',   [])
    print(f'Resumed from {latest_ckpt[-1].name}  (epoch {start_epoch})')
else:
    print('No checkpoint found — training from scratch.')

## 8. Training loop

In [ ]:
def run_epoch(cases: List[AirfRANSCase], train: bool) -> float:
    model.train() if train else model.eval()
    total_loss = 0.0
    if train:
        np.random.shuffle(cases)

    optimizer.zero_grad()

    for step, case in enumerate(cases):
        d = case.load()
        b = make_batch(d['coords'], d['node_input'], d['targets'],
                       CFG['k_neighbors'], node_mean, node_std, out_mean, out_std)

        x          = b['x'].to(DEVICE)
        edge_index = b['edge_index'].to(DEVICE)
        edge_attr  = b['edge_attr'].to(DEVICE)
        targets    = b['y'].to(DEVICE)

        with torch.amp.autocast('cuda', dtype=dtype):
            pred = model(x, edge_index, edge_attr)
            loss = nn.functional.mse_loss(pred, targets)

        if train:
            scaler.scale(loss / CFG['grad_accum']).backward()
            if (step + 1) % CFG['grad_accum'] == 0 or step == len(cases) - 1:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

        total_loss += loss.item()
        del x, edge_index, edge_attr, targets, pred, loss, b, d
        torch.cuda.empty_cache()

    return total_loss / len(cases)


print(f'Starting training from epoch {start_epoch} ...')
print(f'Train: {len(train_cases)} cases  |  Test: {len(test_cases)} cases')
print(f'Epochs: {CFG["epochs"]}  |  Grad accum: {CFG["grad_accum"]}  |  bf16: {CFG["bf16"]}')

t0 = time.time()
for epoch in range(start_epoch, CFG['epochs']):
    t_ep = time.time()

    train_loss = run_epoch(train_cases, train=True)
    scheduler.step()
    train_losses.append(train_loss)

    if (epoch + 1) % CFG['log_every'] == 0 or epoch == 0:
        with torch.no_grad():
            val_loss = run_epoch(test_cases[:20], train=False)
        val_losses.append((epoch, val_loss))
        lr_now  = scheduler.get_last_lr()[0]
        elapsed = time.time() - t_ep
        print(f'Epoch {epoch+1:3d}/{CFG["epochs"]}  '
              f'train={train_loss:.4f}  val={val_loss:.4f}  '
              f'lr={lr_now:.2e}  t={elapsed:.0f}s')

    if (epoch + 1) % CFG['save_every'] == 0:
        ckpt_path = CKPT_DIR / f'ckpt_epoch{epoch+1:04d}.pt'
        torch.save({
            'epoch':        epoch,
            'model':        model.state_dict(),
            'optimizer':    optimizer.state_dict(),
            'scheduler':    scheduler.state_dict(),
            'train_losses': train_losses,
            'val_losses':   val_losses,
            'cfg':          CFG,
        }, ckpt_path)
        print(f'  ✓  Checkpoint → {ckpt_path.name}')

print(f'\nTraining complete in {(time.time()-t0)/60:.1f} min')

## 9. Loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='train MSE')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE (normalised)')
axes[0].set_title('Training Loss'); axes[0].legend()
axes[0].set_yscale('log')

if val_losses:
    ep, vl = zip(*val_losses)
    axes[1].plot(ep, vl, 'o-', color='darkorange', label='val MSE')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE (normalised)')
    axes[1].set_title('Validation Loss'); axes[1].legend()
    axes[1].set_yscale('log')

plt.tight_layout()
plot_path = CKPT_DIR / 'loss_curves.png'
plt.savefig(str(plot_path), dpi=120)
plt.show()
print(f'Saved to {plot_path}')

## 10. Save final checkpoint + summary

In [ ]:
final_path = CKPT_DIR / 'flow_head_final.pt'
torch.save({
    'epoch':        CFG['epochs'] - 1,
    'model':        model.state_dict(),
    'cfg':          CFG,
    'norm':         {
        'node_mean': node_mean.tolist(), 'node_std': node_std.tolist(),
        'out_mean':  out_mean.tolist(),  'out_std':  out_std.tolist(),
    },
    'final_train_loss': train_losses[-1] if train_losses else None,
    'final_val_loss':   val_losses[-1][1] if val_losses else None,
}, final_path)

print(f'Final checkpoint → {final_path}')
print(f'Final train MSE : {train_losses[-1]:.4f}')
if val_losses:
    print(f'Final val MSE   : {val_losses[-1][1]:.4f}')
print('\n✓  Flow head pretraining complete — ready for 03_heat_head.ipynb')